# Notebook 13 — Generazione di immagini programmatica

**Modulo:** 13 — Come funzionano davvero gli LLM
**Tempo stimato:** 1,5-2 ore

## Cosa costruirai

Al termine di questo notebook avrai:
- Una funzione che genera un'immagine da un prompt testuale via API e la salva su disco
- Un piccolo esperimento sull'effetto del livello di dettaglio del prompt
- Un'immagine e una maschera di esempio generate via codice (per non dipendere da file esterni)
- Un esercizio di editing con maschera (inpainting) su quell'immagine di esempio
- Una funzione di stima del costo, sullo stesso principio della `calcola_costo` vista nei notebook precedenti

## Prerequisiti

```bash
pip install openai python-dotenv pillow
```

Serve un file `.env` nella root del progetto con:
```
OPENAI_API_KEY=sk-...
```

> **Nota:** le celle che chiamano l'API generano costi reali (contenuti nell'ordine di pochi centesimi
> a immagine). Se preferisci non spendere, leggi comunque il codice: la struttura e i commenti
> spiegano cosa succederebbe a ogni passo.


---
## Cella 0 — Setup: caricamento delle chiavi

Come nei notebook precedenti, verifichiamo che la chiave sia disponibile prima di procedere.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

OPENAI_KEY = os.getenv("OPENAI_API_KEY")

def verifica_chiave(nome: str, chiave: str | None) -> None:
    if chiave:
        print(f"OK {nome}: {chiave[:8]}...")
    else:
        print(f"MANCANTE {nome}: non trovata. Controlla il tuo file .env")

verifica_chiave("OPENAI_API_KEY", OPENAI_KEY)


---
## Cella 1 — Prima generazione di immagine da testo

La chiamata più semplice: un prompt in ingresso, un'immagine in uscita. Il modello `gpt-image-1`
restituisce l'immagine codificata in base64 (non un URL), quindi dobbiamo decodificarla e salvarla
noi su disco.

In [ ]:
from openai import OpenAI
import base64

client = OpenAI(api_key=OPENAI_KEY)

def genera_immagine(prompt: str, nome_file: str, size: str = "1024x1024", quality: str = "high") -> str:
    """Genera un'immagine da un prompt testuale e la salva su disco. Ritorna il percorso del file."""
    risultato = client.images.generate(
        model="gpt-image-1",  # verifica l'ID modello aggiornato nella doc ufficiale
        prompt=prompt,
        size=size,
        quality=quality,
        n=1
    )
    immagine_bytes = base64.b64decode(risultato.data[0].b64_json)
    with open(nome_file, "wb") as f:
        f.write(immagine_bytes)
    print(f"Immagine salvata in: {nome_file}")
    return nome_file

# Prima prova
genera_immagine(
    prompt="Un gatto arancione seduto su un davanzale, luce del mattino, stile fotorealistico",
    nome_file="gatto.png"
)


---
## Cella 2 — Esperimento: quanto conta il dettaglio del prompt?

Generiamo due immagini a partire dallo stesso soggetto, ma con un livello di dettaglio molto diverso
nel prompt. Confronta visivamente i risultati (apri i due file PNG generati).

**Prima di eseguire:** cosa ti aspetti che cambi tra le due immagini? Solo lo stile, o anche il
soggetto stesso potrebbe variare in modo più marcato con il prompt generico?

In [ ]:
PROMPT_GENERICO = "Una macchina rossa"

PROMPT_DETTAGLIATO = (
    "Una berlina rossa metallizzata, vista di tre quarti anteriore, "
    "ferma a un semaforo in una strada di citta al tramonto, luce calda "
    "che si riflette sulla carrozzeria, sfondo leggermente sfocato con "
    "edifici moderni, stile fotografia editoriale."
)

genera_immagine(PROMPT_GENERICO, "macchina_generica.png")
genera_immagine(PROMPT_DETTAGLIATO, "macchina_dettagliata.png")

print("\nApri i due file e confrontali. Nota quanto varia l'inquadratura, la luce e lo stile.")


**Le tue osservazioni** (modifica questa cella):

- Con il prompt generico, il modello ha scelto liberamente: ...
- Con il prompt dettagliato, il risultato si avvicina di piu a: ...
- Le leve che ho notato fare piu differenza (soggetto, ambientazione, luce, inquadratura, stile): ...

---
## Cella 3 — Stima del costo di una chiamata

Come per le chiamate testuali (vedi Modulo 08), prima di generare immagini in produzione conviene
stimare il costo. Qui i prezzi non dipendono dai token ma da risoluzione, qualita e numero di
immagini richieste.

> Le cifre seguenti sono **puramente illustrative**: verifica sempre il listino aggiornato sul sito
> del provider prima di stimare il costo reale di un progetto.

In [ ]:
# Prezzi ILLUSTRATIVI in USD per immagine (verifica sempre i prezzi reali aggiornati)
PREZZI_IMMAGINE = {
    ("gpt-image-1", "1024x1024", "standard"): 0.04,
    ("gpt-image-1", "1024x1024", "high"):     0.08,
    ("gpt-image-1", "1536x1024", "high"):     0.12,
}

def calcola_costo_immagine(modello: str, size: str, quality: str, n_immagini: int = 1) -> dict:
    """Stima il costo di n immagini generate con una data combinazione modello/size/quality.
    Segue lo stesso principio della funzione calcola_costo usata per le chiamate testuali:
    prezzo unitario per il parametro rilevante, moltiplicato per la quantita richiesta."""
    chiave = (modello, size, quality)
    if chiave not in PREZZI_IMMAGINE:
        raise ValueError(f"Combinazione non in tabella: {chiave}")
    prezzo_unitario = PREZZI_IMMAGINE[chiave]
    costo_totale = prezzo_unitario * n_immagini
    return {
        "modello": modello,
        "size": size,
        "quality": quality,
        "n_immagini": n_immagini,
        "prezzo_unitario_usd": prezzo_unitario,
        "costo_totale_usd": round(costo_totale, 4)
    }

# Esempio: 3.000 immagini standard + 500 in alta qualita al mese (vedi domanda 4 della dispensa)
costo_standard = calcola_costo_immagine("gpt-image-1", "1024x1024", "standard", n_immagini=3000)
costo_alta_qualita = calcola_costo_immagine("gpt-image-1", "1024x1024", "high", n_immagini=500)

print("Standard:", costo_standard)
print("Alta qualita:", costo_alta_qualita)
print(f"\nCosto mensile totale stimato: ${costo_standard['costo_totale_usd'] + costo_alta_qualita['costo_totale_usd']:.2f}")


---
## Cella 4 — Preparare un'immagine e una maschera di esempio

L'editing con maschera richiede un'immagine originale e una maschera con le aree trasparenti nei
punti da modificare. Invece di dipendere da file esterni, generiamo entrambe con `Pillow`, cosi
l'esercizio funziona su qualunque macchina senza bisogno di scaricare nulla.

Disegniamo un tavolo semplice (rettangoli e linee) e creiamo una maschera con un cerchio
trasparente al centro del piano del tavolo: e li che chiederemo al modello di aggiungere qualcosa.

In [ ]:
from PIL import Image, ImageDraw

def crea_immagine_esempio(nome_file: str = "tavolo.png", dimensione: int = 1024) -> str:
    """Disegna un tavolo di legno molto semplice su sfondo chiaro, in PNG quadrato."""
    img = Image.new("RGB", (dimensione, dimensione), color=(245, 240, 230))
    draw = ImageDraw.Draw(img)

    # Piano del tavolo (rettangolo marrone)
    piano = (dimensione * 0.15, dimensione * 0.35, dimensione * 0.85, dimensione * 0.55)
    draw.rectangle(piano, fill=(120, 80, 45))

    # Quattro gambe (linee spesse)
    gamba_larghezza = int(dimensione * 0.03)
    for x in (dimensione * 0.20, dimensione * 0.78):
        draw.rectangle(
            (x, dimensione * 0.55, x + gamba_larghezza, dimensione * 0.85),
            fill=(90, 60, 35)
        )

    img.save(nome_file, "PNG")
    print(f"Immagine di esempio salvata in: {nome_file}")
    return nome_file


def crea_maschera_esempio(
    nome_file: str = "tavolo_maschera.png",
    dimensione: int = 1024,
    centro_relativo: tuple = (0.5, 0.44),
    raggio_relativo: float = 0.12
) -> str:
    """Crea una maschera: opaca ovunque (non modificare), trasparente in un cerchio
    al centro del piano del tavolo (qui il modello potra generare contenuto nuovo)."""
    maschera = Image.new("RGBA", (dimensione, dimensione), color=(255, 255, 255, 255))
    draw = ImageDraw.Draw(maschera)

    cx, cy = centro_relativo[0] * dimensione, centro_relativo[1] * dimensione
    r = raggio_relativo * dimensione
    # Area trasparente = zona che il modello puo modificare
    draw.ellipse((cx - r, cy - r, cx + r, cy + r), fill=(0, 0, 0, 0))

    maschera.save(nome_file, "PNG")
    print(f"Maschera di esempio salvata in: {nome_file}")
    return nome_file


percorso_immagine = crea_immagine_esempio()
percorso_maschera = crea_maschera_esempio()


---
## Cella 5 — Editing con maschera (inpainting)

Ora chiamiamo l'endpoint di editing. Ricorda la regola della dispensa (Sezione 4.2): il prompt deve
descrivere **l'immagine risultante intera**, non solo l'oggetto da inserire nella zona trasparente.

In [ ]:
def modifica_immagine_con_maschera(
    percorso_immagine: str,
    percorso_maschera: str,
    prompt: str,
    nome_file_output: str,
    size: str = "1024x1024"
) -> str:
    """Applica un editing con maschera: modifica solo l'area trasparente della maschera."""
    risultato = client.images.edit(
        model="gpt-image-1",
        image=open(percorso_immagine, "rb"),
        mask=open(percorso_maschera, "rb"),
        prompt=prompt,
        size=size
    )
    immagine_bytes = base64.b64decode(risultato.data[0].b64_json)
    with open(nome_file_output, "wb") as f:
        f.write(immagine_bytes)
    print(f"Immagine modificata salvata in: {nome_file_output}")
    return nome_file_output


modifica_immagine_con_maschera(
    percorso_immagine=percorso_immagine,
    percorso_maschera=percorso_maschera,
    prompt="Un tavolo di legno con sopra una fruttiera piena di frutta fresca, illuminazione naturale",
    nome_file_output="tavolo_con_frutta.png"
)


**Esercizio — `# TODO`:** usando `crea_maschera_esempio`, genera una nuova maschera che renda
trasparente una zona diversa dell'immagine (per esempio, uno dei due angoli superiori dello sfondo
invece del centro del tavolo). Poi chiama `modifica_immagine_con_maschera` con un prompt coerente
con quella nuova zona (per esempio, per aggiungere una finestra sullo sfondo).

In [ ]:
# TODO: crea una maschera alternativa cambiando centro_relativo e raggio_relativo
# maschera_alternativa = crea_maschera_esempio(
#     nome_file="tavolo_maschera_2.png",
#     centro_relativo=(...),
#     raggio_relativo=...
# )

# TODO: chiama modifica_immagine_con_maschera con un prompt che descriva l'immagine risultante intera
# modifica_immagine_con_maschera(
#     percorso_immagine=percorso_immagine,
#     percorso_maschera=maschera_alternativa,
#     prompt="...",
#     nome_file_output="tavolo_variante_2.png"
# )


---
## Cella 6 — Variazioni via editing guidato

Come spiegato nella dispensa (Sezione 4.3), l'endpoint dedicato alle "variazioni" di DALL·E 2 e oggi
in gran parte sostituito dall'editing guidato via prompt: si passa l'immagine originale senza
maschera (o con una maschera che copre tutta l'immagine) e si descrive testualmente lo stile o la
variazione desiderata.

In [ ]:
def genera_variazione_stilistica(percorso_immagine: str, istruzione_stile: str, nome_file_output: str) -> str:
    """Genera una variante dell'immagine originale guidata da un'istruzione di stile testuale,
    usando l'endpoint di editing senza una maschera specifica (si modifica l'immagine intera)."""
    risultato = client.images.edit(
        model="gpt-image-1",
        image=open(percorso_immagine, "rb"),
        prompt=istruzione_stile,
        size="1024x1024"
    )
    immagine_bytes = base64.b64decode(risultato.data[0].b64_json)
    with open(nome_file_output, "wb") as f:
        f.write(immagine_bytes)
    print(f"Variazione salvata in: {nome_file_output}")
    return nome_file_output


genera_variazione_stilistica(
    percorso_immagine=percorso_immagine,
    istruzione_stile="La stessa scena, ma disegnata come un'illustrazione ad acquerello",
    nome_file_output="tavolo_acquerello.png"
)


---
## Sfida finale — Generatore di varianti di prodotto

Immagina di dover fornire a un piccolo e-commerce uno strumento che genera piu varianti visive di
un prodotto a partire da un'unica descrizione testuale di base, per testare quale funziona meglio
in una scheda prodotto.

Costruisci una funzione `genera_varianti_prodotto` che:

1. Riceva una descrizione di base del prodotto (es. "una tazza di ceramica bianca con manico") e
   una lista di "varianti di stile" (es. `["su sfondo bianco da studio fotografico", "su un tavolo
   di legno rustico", "in mano a una persona, sfondo sfocato"]`).
2. Per ogni variante, componga un prompt combinando descrizione + variante di stile e generi
   un'immagine con `genera_immagine`, salvando ogni file con un nome distinto (es.
   `prodotto_variante_1.png`, `prodotto_variante_2.png`, ...).
3. Usi `calcola_costo_immagine` per stimare e stampare il costo totale del batch generato.
4. Ritorni la lista dei percorsi dei file generati.

Completa i `# TODO` qui sotto.

In [ ]:
def genera_varianti_prodotto(descrizione_base: str, varianti_stile: list[str], size: str = "1024x1024", quality: str = "standard") -> list[str]:
    """Genera una immagine per ogni variante di stile, combinando la descrizione base con lo stile."""
    percorsi = []

    for i, stile in enumerate(varianti_stile, start=1):
        # TODO: componi il prompt combinando descrizione_base e stile
        prompt = None  # sostituisci con una f-string che unisce descrizione_base e stile

        # TODO: chiama genera_immagine con il prompt e un nome file distinto per ogni variante
        nome_file = f"prodotto_variante_{i}.png"
        # percorso = genera_immagine(prompt, nome_file, size=size, quality=quality)
        # percorsi.append(percorso)

    # TODO: calcola il costo totale del batch con calcola_costo_immagine e stampalo
    # costo = calcola_costo_immagine("gpt-image-1", size, quality, n_immagini=len(varianti_stile))
    # print(f"Costo stimato del batch: ${costo['costo_totale_usd']:.4f}")

    return percorsi


# Prova la tua implementazione (decommenta quando hai completato i TODO sopra)
# genera_varianti_prodotto(
#     descrizione_base="Una tazza di ceramica bianca con manico",
#     varianti_stile=[
#         "su sfondo bianco da studio fotografico, luce soffusa",
#         "su un tavolo di legno rustico, luce calda pomeridiana",
#         "in mano a una persona, sfondo sfocato, stile lifestyle"
#     ]
# )


---
## Autovalutazione della settimana

Rispondi in questa cella prima di chiudere il notebook:

1. **Cosa ha funzionato bene questa settimana?**
   ...

2. **Cosa non hai capito del tutto?**
   ...

3. **Cosa cambieresti nel tuo approccio la prossima settimana?**
   ...

4. **Tempo reale dedicato:**
   ...

---
*Notebook 13 — Modulo 13 del Corso Completo di IA Generativa*